In [5]:
from pathlib import Path
import sys
import importlib

PROJECT_ROOT = Path("/cluster/tufts/cglab/mcroning/lc_soliton_prod")

sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "legacy"))

# ------------------------------------------------------------------------------------------------------------------------------------

# IMPORT MODULES
# ---------------------------------------------------------

import lc_core.config
import lc_core.pipeline
import lc_core.launch
import lc_core.static_z_march
import lc_offload_tools
import lc_core.io_movies
import lc_core.compare_runs
import lc_core.run_loader

# ---------------------------------------------------------
# RELOAD DURING DEVELOPMENT
# ---------------------------------------------------------

importlib.reload(lc_core.config)
importlib.reload(lc_core.pipeline)
importlib.reload(lc_core.launch)
importlib.reload(lc_core.static_z_march)
importlib.reload(lc_core.io_movies)
importlib.reload(lc_offload_tools)
importlib.reload(lc_core.compare_runs)
importlib.reload(lc_core.run_loader)

# ---------------------------------------------------------
# PUBLIC IMPORTS
# ---------------------------------------------------------

from lc_core.config import (
    config_from_prdata,
    validate_config,
    print_config_summary,
)

from lc_core.pipeline import (
    run_td_experiment,
    run_static_experiment,
)

from lc_core.launch import (
    build_launch_field_gpu,
)

from lc_offload_tools import (
    build_theta_bias_IC,
    build_theta_bias_IC_dirichlet_value,
    compute_n_bg_from_bias,
    genrot,
    build_amp_pair,
    intens,
    run_unified_td_static,
)

from lc_offload_tools import (
    choose_optics_substeps,
    get_h_for_dz,
    prepare_cn_ky_operator,
    strict_static_relax_slice_selfconsistent,
)

from lc_core.io_movies import save_td_movies

from lc_core.compare_runs import (
    save_static_td_comparison,
)

from lc_core.run_loader import (
    load_config_dict,
    load_prdata,
    rebuild_cfg,
    rebuild_ctx,
    load_array,
    load_saved_result,
    load_movie,
    list_run_contents,
)


In [13]:
td_run_dir = Path("/cluster/tufts/cglab/mcroning/lc_runs/true_td_20260516_094855")

list_run_contents(td_run_dir)

{'run_dir': '/cluster/tufts/cglab/mcroning/lc_runs/true_td_20260516_094855',
 'files': ['config.json', 'environment.json', 'td_summary.json'],
 'arrays': ['arrays/I_mid_store.npy',
  'arrays/array_manifest.json',
  'arrays/theta_full.npy'],
 'movies': [],
 'comparisons': ['comparisons/.ipynb_checkpoints',
  'comparisons/I_mid_diff_k0000.png',
  'comparisons/I_mid_diff_k0025.png',
  'comparisons/I_mid_diff_k0050.png',
  'comparisons/I_mid_diff_k0075.png',
  'comparisons/I_mid_diff_k0099.png',
  'comparisons/I_mid_diff_vs_z.png',
  'comparisons/static_vs_td_summary.json',
  'comparisons/theta_diff_k0000.png',
  'comparisons/theta_diff_k0025.png',
  'comparisons/theta_diff_k0050.png',
  'comparisons/theta_diff_k0075.png',
  'comparisons/theta_diff_k0099.png',
  'comparisons/theta_diff_vs_z.png'],
 'diagnostics': []}

In [14]:
td_loaded = load_saved_result(td_run_dir, xp=None)
Iyz = load_movie(td_run_dir, "arrays/theta_full.npy")

FileNotFoundError: [Errno 2] No such file or directory: '/arrays/theta_full.npy.npy'

## prdata

In [2]:
# 1. Edit this cell only

prdata = dict(
    xsamp=512,
    ysamp=512,
    yaper=100.0,
    xaper=75,
    rlen=3000.0,
    dz=5.0,

    d=75.0,
    K=7e-12,
    De=13.0,
    bias_voltage=1.1,
    ne=1.7,
    no=1.5,

    P=0.5,

    w0x1=3.0,
    w0y1=3.0,
    w0x2=3.0,
    w0y2=3.0,

    thout1=0.0,
    thout2=-0.0,
    phi1=0.0,
    phi2=0.0,

    xoffset=0.0,
    yoffset=0.0,
    soliton_pair_sep=0.0,
    soliton_pair_angle=90.0,

    coh=False,
    soliton=True,

    time_behavior="Time Dependent",
    tend=0.02,
    tsteps=10,
    t_stride=2,
    save_full_I_mid_store=True
    
)

In [38]:
prdata["time_behavior"] = "Static"
prdata["rlen"] = 500.0
prdata["dz"] = 5.0

## Build config

In [39]:
cfg = config_from_prdata(prdata)
validate_config(cfg)
print_config_summary(cfg)

LC RunConfig summary
--------------------
Grid:    Nx=512, Ny=512, Nz=100
Spacing: dx=0.146484 um, dy=0.195312 um, dz=5 um
Aperture: x=75 um, y=100 um, z=500 um
LC:      b=2.48703, bi=428.571, ne=1.7, no=1.5
Bias:    V=1.1, theta_bc=0
Launch:  coh=False, thout=(0.0, -0.0), sep=0.0 um
Time:    timedep=False, tsteps=1, dt=0.02
Sponge:  enabled=True, windowedge=0.1


In [35]:
# Run TD
ctx, stores, launch_arrays, td_res, run_dir = run_td_experiment(
    cfg,
    run_root="/cluster/tufts/cglab/mcroning/lc_runs",

    build_context_kwargs=dict(
        build_theta_bias_IC=build_theta_bias_IC,
        build_theta_bias_IC_dirichlet_value=build_theta_bias_IC_dirichlet_value,
        compute_n_bg_from_bias=compute_n_bg_from_bias,
        build_launch_field_gpu=build_launch_field_gpu,
        genrot=genrot,
        build_amp_pair=build_amp_pair,
        intens=intens,
    ),

    td_bridge_kwargs=dict(
        run_unified_td_static=run_unified_td_static,
        report_runtime=True,
        tqdm_timedep=False,
        tqdm_static_z=False,
    ),
)

td_res.summary

DEBUG waist keys: 3.0 3.0 3.0 3.0
Initialized GPU context:
  Nx,Ny,Nz = 512, 512, 600
  dx,dy,dz = 0.146484, 0.195312, 5 um
  refin    = 1.59826899
  b, bi    = 2.48702536, 428.571429
  coh      = False
  amp0     = shape (2, 512, 512)
  theta    = shape (600, 512, 512), dtype float32
  approx allocated in ctx/stores = 1.202 GiB
  timedep  = True, tsteps=10, dt=0.002
  slice movies = True
[theta stride] theta_z_stride_um=0.0 -> theta_k_stride=1 slices
[optics substep] dz=5.0 um phi_est=0.993 -> Nsub=2 dz_sub=2.5 um
[theta z coupling] theta_z_gamma=0.0
[TD mode] true physical-time integrator


In [10]:
td_res.summary

NameError: name 'td_res' is not defined

In [37]:
print(run_dir)
print(list((run_dir / "arrays").glob("*")))
print(list(run_dir.glob("*summary*.json")))

/cluster/tufts/cglab/mcroning/lc_runs/true_td_20260516_094855
[PosixPath('/cluster/tufts/cglab/mcroning/lc_runs/true_td_20260516_094855/arrays/I_mid_store.npy'), PosixPath('/cluster/tufts/cglab/mcroning/lc_runs/true_td_20260516_094855/arrays/array_manifest.json'), PosixPath('/cluster/tufts/cglab/mcroning/lc_runs/true_td_20260516_094855/arrays/theta_full.npy')]
[PosixPath('/cluster/tufts/cglab/mcroning/lc_runs/true_td_20260516_094855/td_summary.json')]


In [31]:
movie_manifest = save_td_movies(
    td_res,
    run_dir,
)

movie_manifest

saved movies to: /cluster/home/mcroning/cglab/mcroning/true_td_20260516_092906/movies


In [32]:
run_dir

PosixPath('/cluster/home/mcroning/cglab/mcroning/true_td_20260516_092906')

In [45]:
ctx_s, stores_s, launch_s, static_res, static_run_dir = run_static_experiment(
    cfg,
    run_root="/cluster/tufts/cglab/mcroning/lc_runs",

    build_context_kwargs=dict(
        build_theta_bias_IC=build_theta_bias_IC,
        build_theta_bias_IC_dirichlet_value=build_theta_bias_IC_dirichlet_value,
        compute_n_bg_from_bias=compute_n_bg_from_bias,
        build_launch_field_gpu=build_launch_field_gpu,
        genrot=genrot,
        build_amp_pair=build_amp_pair,
        intens=intens,
    ),

    static_bridge_kwargs=dict(
        choose_optics_substeps=choose_optics_substeps,
        get_h_for_dz_legacy=get_h_for_dz,
        prepare_cn_ky_operator=prepare_cn_ky_operator,
        strict_static_relax_slice_selfconsistent=strict_static_relax_slice_selfconsistent,
        strict_residual_tol_max=6e-2,
        strict_residual_tol_rms=1e-2,
        strict_max_outer_passes=8,
        verbose_every=25,
        tqdm_z=False,
    ),
)

static_res.summary, static_run_dir

DEBUG waist keys: 3.0 3.0 3.0 3.0
Initialized GPU context:
  Nx,Ny,Nz = 512, 512, 100
  dx,dy,dz = 0.146484, 0.195312, 5 um
  refin    = 1.59826899
  b, bi    = 2.48702536, 428.571429
  coh      = False
  amp0     = shape (2, 512, 512)
  theta    = shape (100, 512, 512), dtype float32
  approx allocated in ctx/stores = 0.203 GiB
  timedep  = False, tsteps=1, dt=0.02
  slice movies = True
[strict z bridge] Nsub=2 dz_sub=2.5 phi_est=0.993 Nz=100
[k=   0] conv=False self=3 max=5.203e-02 rms=9.084e-03
[k=  25] conv=True self=3 max=5.380e-02 rms=9.118e-03
[k=  50] conv=True self=3 max=4.999e-02 rms=9.182e-03
[k=  75] conv=True self=3 max=5.847e-02 rms=9.077e-03
[k=  99] conv=True self=3 max=5.501e-02 rms=9.136e-03


({'mode': 'strict_static_z_march_bridge',
  'Nz': 100,
  'dz_um': 5.0,
  'Nsub': 2,
  'dz_sub_um': 2.5,
  'phi_est': 0.9926043139304243,
  'converged_count': 99,
  'failed_count': 1,
  'max_residual_max': 0.05892930859069123,
  'max_residual_rms': 0.009197970830308171,
  'P_final': 0.9999449253082275},
 PosixPath('/cluster/tufts/cglab/mcroning/lc_runs/strict_static_20260516_100439'))

In [46]:
print(static_run_dir)
print(list(static_run_dir.glob("*")))
print(list((static_run_dir / "pde_defect_diagnostics").glob("*"))[:5])
print(list((static_run_dir / "arrays").glob("*")))

/cluster/tufts/cglab/mcroning/lc_runs/strict_static_20260516_100439
[PosixPath('/cluster/tufts/cglab/mcroning/lc_runs/strict_static_20260516_100439/arrays'), PosixPath('/cluster/tufts/cglab/mcroning/lc_runs/strict_static_20260516_100439/config.json'), PosixPath('/cluster/tufts/cglab/mcroning/lc_runs/strict_static_20260516_100439/environment.json'), PosixPath('/cluster/tufts/cglab/mcroning/lc_runs/strict_static_20260516_100439/pde_defect_diagnostics'), PosixPath('/cluster/tufts/cglab/mcroning/lc_runs/strict_static_20260516_100439/static_z_reports.json'), PosixPath('/cluster/tufts/cglab/mcroning/lc_runs/strict_static_20260516_100439/static_z_summary.json')]
[PosixPath('/cluster/tufts/cglab/mcroning/lc_runs/strict_static_20260516_100439/pde_defect_diagnostics/pde_defect_k0000.png'), PosixPath('/cluster/tufts/cglab/mcroning/lc_runs/strict_static_20260516_100439/pde_defect_diagnostics/pde_defect_k0025.png'), PosixPath('/cluster/tufts/cglab/mcroning/lc_runs/strict_static_20260516_100439/pde_

In [9]:
res_static_loaded = saved_arrays_as_result(
    static_run_dir,
    xp=None,
)

comparison = save_static_td_comparison(
    ctx_s,
    res_static_loaded,
    td_res,
    run_dir,
)

comparison["theta_stats"]

NameError: name 'ctx_s' is not defined

In [6]:
from pathlib import Path

static_run_dir = Path(
    "/cluster/tufts/cglab/mcroning/lc_runs/strict_static_20260516_100439"
)